# Preparación del conjunto de datos

En este notebook se muestran algunas de las técnicas más utilizadas para transformar el conjunto de datos.

## Conjunto de datos

### Descripción
NSL-KDD es un conjunto de datos propuesto para resolver algunos de los problemas inherentes del conjunto de datos KDD'99. Sigue siendo un benchmark efectivo para comparar métodos de detección de intrusiones.

### Ficheros de datos utilizados
* **KDDTrain+.csv**: Conjunto de entrenamiento completo NSL-KDD en formato CSV

### Referencias
_M. Tavallaee, E. Bagheri, W. Lu, and A. Ghorbani, "A Detailed Analysis of the KDD CUP 99 Data Set," Submitted to Second IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA), 2009._

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Funciones auxiliares

In [ ]:
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty_level'
]

NORMAL_LABEL = 'normal'


def load_kdd_dataset(data_path):
    """Lectura del conjunto de datos NSL-KDD desde CSV."""
    df = pd.read_csv(data_path, header=None, names=COLUMN_NAMES)
    df.drop('difficulty_level', axis=1, inplace=True)
    df['class'] = df['class'].apply(lambda x: NORMAL_LABEL if x == NORMAL_LABEL else 'anomaly')
    for col in df.select_dtypes(exclude=['object']).columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [ ]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

## 1. Lectura del conjunto de datos

In [ ]:
df = load_kdd_dataset("NSL_KDD-master/KDDTrain+.csv")

In [ ]:
df

## 2. División del conjunto de datos

In [ ]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

In [ ]:
print("Longitud del Training Set:", len(train_set))
print("Longitud del Validation Set:", len(val_set))
print("Longitud del Test Set:", len(test_set))

## 3. Limpiando los datos

Antes de comenzar, vamos a recuperar el conjunto de datos limpio y vamos a separar las etiquetas del resto de los datos, no necesariamente queremos aplicar las mismas transformaciones en ambos conjuntos.

In [ ]:
# Separamos las características de entrada de la característica de salida
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [ ]:
# Para ilustrar esta sección vamos a añadir algunos valores nulos
# a algunas características del conjunto de datos
X_train = X_train.copy()
X_train.loc[(X_train["src_bytes"] > 400) & (X_train["src_bytes"] < 800), "src_bytes"] = np.nan
X_train.loc[(X_train["dst_bytes"] > 500) & (X_train["dst_bytes"] < 2000), "dst_bytes"] = np.nan
X_train

La mayoría de los algoritmos de Machine Learning no pueden trabajar sobre características que contengan valores nulos. Por ello, existen tres opciones para reemplazarlos:
* Eliminar las filas correspondientes
* Eliminar el atributo (columna) correspondiente
* Rellenarlos con un valor determinado (zero, media ...)

In [ ]:
# Comprobamos si existe algún atributo con valores nulos
X_train.isna().any()

In [ ]:
# Seleccionamos las filas que contienen valores nulos
filas_valores_nulos = X_train[X_train.isnull().any(axis=1)]
filas_valores_nulos

### Opción 1: Eliminamos las filas con valores nulos

In [ ]:
# Copiamos el conjunto de datos para no alterar el original
X_train_copy = X_train.copy()

In [ ]:
# Eliminamos las filas con valores nulos
X_train_copy.dropna(subset=["src_bytes", "dst_bytes"], inplace=True)
X_train_copy

In [ ]:
# Contamos el número de filas eliminadas
print("El número de filas eliminadas es:", len(X_train) - len(X_train_copy))

### Opción 2: Eliminamos los atributos con valores nulos

In [ ]:
# Copiamos el conjunto de datos para no alterar el original
X_train_copy = X_train.copy()

In [ ]:
# Eliminamos los atributos con valores nulos
X_train_copy.drop(["src_bytes", "dst_bytes"], axis=1, inplace=True)
X_train_copy

In [ ]:
# Contamos el número de atributos eliminados
print("El número de atributos eliminados es:", len(list(X_train)) - len(list(X_train_copy)))

### Opción 3: Rellenamos los valores nulos con un valor determinado

In [ ]:
# Copiamos el conjunto de datos para no alterar el original
X_train_copy = X_train.copy()

In [ ]:
# Rellenamos los valores nulos con la media de los valores del atributo
media_srcbytes = X_train_copy["src_bytes"].mean()
media_dstbytes = X_train_copy["dst_bytes"].mean()

X_train_copy["src_bytes"] = X_train_copy["src_bytes"].fillna(media_srcbytes)
X_train_copy["dst_bytes"] = X_train_copy["dst_bytes"].fillna(media_dstbytes)

X_train_copy

In [ ]:
# Copiamos el conjunto de datos para no alterar el original
X_train_copy = X_train.copy()

In [ ]:
# Un valor muy alto en el atributo puede disparar la media
# Rellenamos los valores con la mediana
mediana_srcbytes = X_train_copy["src_bytes"].median()
mediana_dstbytes = X_train_copy["dst_bytes"].median()

X_train_copy["src_bytes"] = X_train_copy["src_bytes"].fillna(mediana_srcbytes)
X_train_copy["dst_bytes"] = X_train_copy["dst_bytes"].fillna(mediana_dstbytes)

X_train_copy

### Existe otra alternativa para la opción 3 que consiste en usar la clase Imputer de sklearn

In [ ]:
# Copiamos el conjunto de datos para no alterar el original
X_train_copy = X_train.copy()

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

In [ ]:
# La clase imputer no admite valores categóricos, eliminamos los atributos categóricos
X_train_copy_num = X_train_copy.select_dtypes(exclude=['object'])
X_train_copy_num.info()

In [ ]:
# Se le proporcionan los atributos numéricos para que calcule los valores
imputer.fit(X_train_copy_num)

In [ ]:
# Rellenamos los valores nulos
X_train_copy_num_nonan = imputer.transform(X_train_copy_num)

In [ ]:
# Transformamos el resultado a un DataFrame de Pandas
X_train_copy = pd.DataFrame(X_train_copy_num_nonan, columns=X_train_copy_num.columns)

In [ ]:
X_train_copy.head(10)

## APIs de sklearn

Antes de continuar vamos a hacer una pequeña reseña sobre como funcionan las APIs de sklearn:
* **Estimators**: Cualquier objeto que puede estimar algún parámetro:
    * El propio estimator se forma mediante el método `fit()`, que siempre toma un dataset como argumento
    * Cualquier otro parámetro de este método, es un hiperparámetro
* **Transformers**: Son estimadores capaces de transformar el conjunto de datos (como Imputer)
    * La transformación se realiza mediante el método `transform()`
    * Reciben un dataset como parámetro de entrada
* **Predictors**: Son estimadores capaces de realizar predicciones
    * La predicción se realiza mediante el método `predict()`
    * Reciben un dataset como entrada
    * Retornan un dataset con las predicciones
    * Tienen un método `score()` para evaluar el resultado de la predicción

## 4. Transformación de atributos categóricos a numéricos

Antes de comenzar, vamos a recuperar el conjunto de datos limpio y vamos a separar las etiquetas del resto de los datos.

In [ ]:
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

Los algoritmos de Machine Learning por norma general, ingieren datos numéricos. En nuestro conjunto de datos tenemos una gran cantidad de valores categóricos y en consecuencia debemos convertirlos a numéricos.

In [ ]:
X_train.info()

Existen diferentes formas de convertir los atributos categóricos en numéricos. Probablemente, la más sencilla es la que proporciona el método **factorize** de Pandas. Que transforma cada categoría en un número secuencial.

In [ ]:
protocol_type = X_train['protocol_type']
protocol_type_encoded, categorias = protocol_type.factorize()

In [ ]:
# Mostramos por pantalla como se han codificado
for i in range(10):
    print(protocol_type.iloc[i], "=", protocol_type_encoded[i])

In [ ]:
print(categorias)

### Transformaciones avanzadas mediante sklearn

#### Ordinal Encoding

Realiza la misma codificación que el método **factorize** de Pandas.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

protocol_type = X_train[['protocol_type']]

ordinal_encoder = OrdinalEncoder()
protocol_type_encoded = ordinal_encoder.fit_transform(protocol_type)

In [ ]:
# Mostramos por pantalla como se han codificado
for i in range(10):
    print(protocol_type["protocol_type"].iloc[i], "=", protocol_type_encoded[i])

In [ ]:
print(ordinal_encoder.categories_)

El problema de este tipo de codificación radica en que ciertos algoritmos de ML que funcionan midiendo la similitud de dos puntos por distancia, van a considerar que el 1 está más cerca del 2 que del 3, y en este caso (para estos valores categóricos), no tiene sentido. Por ello, se utilizan otros métodos de categorización, como, por ejemplo, One-Hot encoding.

#### One-Hot Encoding

Genera para cada categoría del atributo categórico una matriz binaria que representa el valor.

In [ ]:
# La sparse matrix solo almacena la posición de los valores que no son '0' para ahorrar memoria
from sklearn.preprocessing import OneHotEncoder

protocol_type = X_train[['protocol_type']]

oh_encoder = OneHotEncoder()
protocol_type_oh = oh_encoder.fit_transform(protocol_type)
protocol_type_oh

In [ ]:
# Convertir la sparse matrix a un array de Numpy
protocol_type_oh.toarray()

In [ ]:
# Mostramos por pantalla como se han codificado
for i in range(10):
    print(protocol_type["protocol_type"].iloc[i], "=", protocol_type_oh.toarray()[i])

In [ ]:
print(ordinal_encoder.categories_)

En muchas ocasiones al particionar el conjunto de datos o al realizar una predicción con nuevos ejemplos aparecen nuevos valores para determinadas categorías que producirán un error en la función **transform()**. La clase OneHotEncoding proporciona el parámetro **handle_unknown** ya sea para generar un error o ignorar si una característica categórica desconocida está presente durante la transformación (el valor predeterminado es lanzar un error).

Cuando este parámetro se establece en "ignorar" y se encuentra una categoría desconocida durante la transformación, las columnas codificadas resultantes para esta característica serán todo ceros. En la transformación inversa, una categoría desconocida se denotará como None.

In [ ]:
oh_encoder = OneHotEncoder(handle_unknown='ignore')

#### Get Dummies

Get Dummies es un método sencillo de utilizar que permite aplicar One-Hot Encoding a un DataFrame de Pandas.

In [ ]:
pd.get_dummies(X_train['protocol_type'])

## 5. Escalado del conjunto de datos

Antes de comenzar, vamos a recuperar el conjunto de datos limpio y vamos a separar las etiquetas del resto de los datos.

In [ ]:
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

Por norma general, los algoritmos de Machine Learning no se comportan adecuadamente si los valores de las características que reciben como entrada se encuentran en rangos muy dispares. Por ello, se utilizan distintas técnicas de escalado. **Importante tener en cuenta que estos mecanismos de escalado no deben aplicarse sobre las etiquetas.**
* **Normalización:** Los valores del atributo se escalan para adquirir un valor entre 0 y 1
* **Estandarización:** Los valores del atributo se escalan y reciben un valor similar pero no se encuentra dentro de un rango

**Es importante que para probar estos valores se realicen las transformaciones solo sobre el conjunto de datos de entrenamiento. Después, se aplicarán sobre el conjunto de datos de prueba para testear.**

In [ ]:
from sklearn.preprocessing import RobustScaler

scale_attrs = X_train[['src_bytes', 'dst_bytes']]

robust_scaler = RobustScaler()
X_train_scaled = robust_scaler.fit_transform(scale_attrs)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=['src_bytes', 'dst_bytes'])

In [ ]:
X_train_scaled.head(10)

In [ ]:
X_train.head(10)